In [1]:
#!pip install xgboost lightgbm optuna

   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 1.5/1.5 MB 25.3 MB/s  0:00:00
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 2.1/2.1 MB 125.6 MB/s  0:00:00

   ---------- ----------------------------- 2/8 [greenlet]
   -------------------- ------------------- 4/8 [sqlalchemy]
   -------------------- ------------------- 4/8 [sqlalchemy]
   -------------------- ------------------- 4/8 [sqlalchemy]
   -------------------- ------------------- 4/8 [sqlalchemy]
   -------------------- ------------------- 4/8 [sqlalchemy]
   -------------------- ------------------- 4/8 [sqlalchemy]
   ------------------------- -------------- 5/8 [lightgbm]
   ------------------------------ --------- 6/8 [alembic]
   ----------------------------------- ---- 7/8 [optuna]
   ----------------------------------- ---- 7/8 [optuna]
   ----------------------------------- ---- 7/8 [optuna]

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer

from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import cross_val_score

In [4]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import roc_auc_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import RocCurveDisplay

In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

import optuna

print('라이브러리 불러오기 완료')

라이브러리 불러오기 완료


In [8]:
plt.rcParams['font.family'] = 'Malgun Gothic'

plt.rcParams['axes.unicode_minus'] = False

In [9]:
data = load_breast_cancer()

df = pd.DataFrame(data.data, columns = data.feature_names)

In [10]:
df['target_original'] = data.target

In [11]:
df['target'] = 1 - df['target_original']

In [12]:
df['target_name'] = df['target'].map({
    0: 'benign',
    1: 'malignant'
})

In [14]:
print('데이터 크기 (행, 열): ', df.shape)

print('\n원본 target 기준 개수: ')
print(df['target_original'].value_counts().sort_index())

print('\n수업용 target 기준 개수: ')
print(df['target'].value_counts().sort_index())

print('\n수업용 target 이름 기준 개수: ')
print(df['target_name'].value_counts())

데이터 크기 (행, 열):  (569, 33)

원본 target 기준 개수: 
target_original
0    212
1    357
Name: count, dtype: int64

수업용 target 기준 개수: 
target
0    357
1    212
Name: count, dtype: int64

수업용 target 이름 기준 개수: 
target_name
benign       357
malignant    212
Name: count, dtype: int64


In [15]:
X = df.drop(columns = ['target_original' ,'target', 'target_name'])

y = df['target']

print('X 크기: ', X.shape)
print('y 크기: ', y.shape)

X 크기:  (569, 30)
y 크기:  (569,)


In [16]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size = 0.2,
    random_state = 42,
    stratify = y
)

print('train 크기: ', X_train.shape)
print('test 크기: ', X_test.shape)
print('\ntrain의 target 비율: ')
print(y_train.value_counts(normalize = True).round(3))
print('\ntest의 target 비율:')
print(y_test.value_counts(normalize = True).round(3))

train 크기:  (455, 30)
test 크기:  (114, 30)

train의 target 비율: 
target
0    0.626
1    0.374
Name: proportion, dtype: float64

test의 target 비율:
target
0    0.632
1    0.368
Name: proportion, dtype: float64


In [26]:
def evaluate_classification_model(model_name, y_true, y_pred, y_proba):
    accuracy = accuracy_score(y_test, y_pred)
    
    precision_malignant = precision_score(y_true, y_pred, pos_label = 1)    
    
    recall_malignant = recall_score(y_true, y_pred, pos_label = 1)

    f1_malignant = f1_score(y_true, y_pred, pos_label = 1)

    roc_auc = roc_auc_score(y_true, y_proba)

    # TN: 실제 benign이고 bening으로 예측한 경우
    # FP: 실제 benign인데 malignant로 예측한 경우
    # FN: 실제 malignant인데 benign으로 예측한 경우
    # TP 실제 malignant인데 malignant로 예측한 경우
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    result = {
        'model_name': model_name,
        'accuracy': accuracy,
        'precision_malignant': precision_malignant,
        'recall_malignant': recall_malignant,
        'f1_malignant': f1_malignant,
        'roc_auc': roc_auc,
        'TN': tn,
        'FP': fp,
        'FN': fn,
        'TP': tp
    }
    
    return result

print('평가 함수 준비 완료')

평가 함수 준비 완료


In [27]:
xgb_base_model = XGBClassifier(
    n_estimators = 100,
    max_depth = 3,
    learning_rate = 0.1,
    random_state = 42,
    eval_metric = 'logloss'
)

In [28]:
xgb_base_model.fit(X_train, y_train)

xgb_base_pred = xgb_base_model.predict(X_test)

xgb_base_proba = xgb_base_model.predict_proba(X_test)[:, 1]

In [29]:
xgb_base_result = evaluate_classification_model(
    model_name = 'XGBoost Base',
    y_true = y_test,
    y_pred= xgb_base_pred,
    y_proba = xgb_base_proba
)

xgb_base_result

{'model_name': 'XGBoost Base',
 'accuracy': 0.9649122807017544,
 'precision_malignant': 1.0,
 'recall_malignant': 0.9047619047619048,
 'f1_malignant': 0.95,
 'roc_auc': 0.9966931216931217,
 'TN': np.int64(72),
 'FP': np.int64(0),
 'FN': np.int64(4),
 'TP': np.int64(38)}

In [36]:
print('=== Classification REport (XGBoost Base) ===')
print(classification_report(
    y_test,
    xgb_base_pred,
    target_names = ['benign(0)', 'malignent(1)']
))

print('=== Confusion Matrix (XGBoost Base) ===')
print(confusion_matrix(y_test, xgb_base_pred))
print('\n[[TN, FP]]')
print(' [FN, TP]] 순서입니다.')

=== Classification REport (XGBoost Base) ===
              precision    recall  f1-score   support

   benign(0)       0.95      1.00      0.97        72
malignent(1)       1.00      0.90      0.95        42

    accuracy                           0.96       114
   macro avg       0.97      0.95      0.96       114
weighted avg       0.97      0.96      0.96       114

=== Confusion Matrix (XGBoost Base) ===
[[72  0]
 [ 4 38]]

[[TN, FP]]
 [FN, TP]] 순서입니다.


In [52]:
# 튜닝에 사용할 XGBoost(설정값은 비워두고 탐색으로 채울 예정)
xgb_for_random_search = XGBClassifier(
    random_state = 42,
    eval_metric = 'logloss'
)

In [53]:
# 탐색할 하이퍼파라미터 후보 (수업 시간을 고려해 너무 넓지 않게 제한)
param_distributions = {
    'n_estimators': [50, 100, 150, 200],
    'max_depth': [2, 3, 4, 5],
    'learning_rate': [0.01, 0.03, 0.05, 0.1],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'reg_alpha': [0, 0.01, 0.1, 1.0],
    'reg_lambda': [0.5, 1.0, 1.5, 2.0]
}

In [54]:
# 교차검증 방식: 3등분, 섞어서 나눔, 재현성 고정
cv = StratifiedKFold(
    n_splits = 3,
    shuffle = True,
    random_state = 42
)

print('탐색 범위 설정 완료')

탐색 범위 설정 완료


In [55]:
random_search = RandomizedSearchCV(
    estimator = xgb_for_random_search,
    param_distributions = param_distributions,
    n_iter = 20, # 무작위 조합 20개
    scoring = 'roc_auc', # ROC-AUC 기준으로 좋은 조합 찾기
    cv = cv, # 3등분 교차검증
    random_state = 42,
    n_jobs = -1
)

# train 데이터 안에서 탐색을 실행합니다. (text는 사용하지 않음)
random_search.fit(X_train, y_train)

print('RandomizedSearchCV 탐색 완료')

RandomizedSearchCV 탐색 완료


In [56]:
print('Best ROC-AUC: ', random_search.best_score_)
print('Best Params: ', random_search.best_params_)
print('Best Estimator: ', random_search.best_estimator_)

Best ROC-AUC:  0.9908565272831202
Best Params:  {'subsample': 0.7, 'reg_lambda': 1.0, 'reg_alpha': 1.0, 'n_estimators': 100, 'max_depth': 3, 'learning_rate': 0.1, 'colsample_bytree': 0.9}
Best Estimator:  XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.9, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=3, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=100, n_jobs=None,
              num_parallel_tree=None, ...)


In [57]:
# 탐색으로 찾은 가장 좋은 모델 선택
xgb_random_search_model = random_search.best_estimator_

# test 데이터로 예측
xgb_random_search_pred = xgb_random_search_model.predict(X_test)
xgb_random_search_proba = xgb_random_search_model.predict_proba(X_test)[:, 1]

xgb_random_search_result = evaluate_classification_model(
    model_name = 'XGBoost RandomizedSearchCV',
    y_true = y_test,
    y_pred = xgb_random_search_pred,
    y_proba = xgb_random_search_proba
)

xgb_random_search_result

{'model_name': 'XGBoost RandomizedSearchCV',
 'accuracy': 0.9824561403508771,
 'precision_malignant': 1.0,
 'recall_malignant': 0.9523809523809523,
 'f1_malignant': 0.975609756097561,
 'roc_auc': 0.9930555555555555,
 'TN': np.int64(72),
 'FP': np.int64(0),
 'FN': np.int64(2),
 'TP': np.int64(40)}

In [66]:
# Optuna 주요 용어
# - trial: trial 1번 = 한번의 실험, 설정값을 하나 정해서 학습하고 평가하는 것입니다.
# - study: 여러 trial을 모아 관리하는 공간. 여러 번의 실험 기록을 모아둔 실험 노트입니다.
# - objective: Optuna가 반복해서 실행하는 평가 함수. 
#              여러 설정값을 넣어 보고 점수를 확인합니다.
# - best_params/best_value: 가장 좋은 점수를 낸 설정값 조합과 그 점수.
#                           단, 이는 train 내부 교차검증 점수입니다.

def objective(trial):
    """
    Optuna가 XGBoost 하이퍼파라미터를 실험할 때 사용할 함수

    trial: 한 번의 실험을 의미
    Optuna는 trial마다 서로 다른 하이퍼파라미터 값을 넣어보고,
    그 결과 점수가 좋은지 확인합니다.
    """
    # trial_suggest_*: Optuna가 정해진 범위 안에서 값을 하나 골라줍니다.
    params = {
        # 트리 개수: 50~300사이 정수 중 선택
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        # 트리 깊이: 2 ~ 6 사이 정수
        'max_depth': trial.suggest_int('max_depth', 2, 6), 
        # 학습 속도: 0.01 ~ 0.2, log = True는 작은 값 쪽을 더 촘촘히 탐색
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log = True),
        # 행 샘플링 비율: 0.7 ~ 1.0
        'subsample': trial.suggest_float('subsample', 0.7, 1.0),
        # feature 샘플링 비율: 0.7 ~ 1.0
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 1.0),
        # L1 규제: 0.0 ~ 1.0
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 1.0),
        # L2 규제: 0.5 ~ 3.0
        'reg_lambda': trial.suggest_float('reg_lambda', 0.5, 3.0),
        'random_state': 42,
        'eval_metric': 'logloss'
    }

    # 위에서 고른 설정값으로 XGBoost 모델을 만듭니다.
    model = XGBClassifier(**params)
    
    # train 데이터 안에서 3등분 교차검증
    cv = StratifiedKFold(
        n_splits = 3,
        shuffle = True,
        random_state = 42
    )
    
    # 교차검증을 통해 ROC-AUC 점수들을 계산합니다.(test는 사용 안 함)
    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv = cv,
        scoring='roc_auc'
    )

    # 교차검증 점수들의 평균을 이 trial의 점수로 돌려줍니다.
    return scores.mean()

print('objective 함수 준비 완료')

objective 함수 준비 완료


In [67]:
# 로그가 너무 많이 출력되면 수업 집중이 어려워 경고 수준만 보이도록 낮춥니다.
optuna.logging.set_verbosity(optuna.logging.WARNING)

# 재현성을 위한 샘플러
sampler = optuna.samplers.TPESampler(seed = 42)

# 점수를 최대화하는 study 생성
study = optuna.create_study(
    direction = 'maximize',
    sampler = sampler
)

# objective 함수를 20번 실험합니다.
study.optimize(objective, n_trials = 20)

print('Optuna 튜닝 완료')

Optuna 튜닝 완료


In [68]:
print('Best ROC-AUC: ', study.best_value)
print('Best Params: ', study.best_params)

# 각 trial 기록을 표로 확인
study.trials_dataframe().head()

Best ROC-AUC:  0.9911082530888625
Best Params:  {'n_estimators': 294, 'max_depth': 3, 'learning_rate': 0.06729392009628148, 'subsample': 0.77457520756364, 'colsample_bytree': 0.909182130134382, 'reg_alpha': 0.07977803963614018, 'reg_lambda': 2.659229016644547}


,number,value,datetime_start,datetime_complete,duration,params_colsample_bytree,params_learning_rate,params_max_depth,params_n_estimators,params_reg_alpha,params_reg_lambda,params_subsample,state
0,0,0.990299,2026-06-18 17:45:27.853805,2026-06-18 17:45:28.386230,0 days 00:00:00.532425,0.746806,0.089608,6,144,0.155995,0.645209,0.879598,COMPLETE
1,1,0.990432,2026-06-18 17:45:28.386230,2026-06-18 17:45:29.222367,0 days 00:00:00.836137,0.990973,0.083411,5,267,0.832443,1.030848,0.706175,COMPLETE
2,2,0.986328,2026-06-18 17:45:29.222367,2026-06-18 17:45:29.503428,0 days 00:00:00.281061,0.829584,0.024879,2,95,0.291229,2.029632,0.857427,COMPLETE
3,3,0.987573,2026-06-18 17:45:29.503428,2026-06-18 17:45:29.807994,0 days 00:00:00.304566,0.935553,0.029967,3,85,0.199674,1.785586,0.836821,COMPLETE
4,4,0.990421,2026-06-18 17:45:29.807994,2026-06-18 17:45:30.334777,0 days 00:00:00.526783,0.719515,0.061721,2,198,0.948886,2.914080,0.751157,COMPLETE


In [69]:
# best_params에 재현성/평가 설정을 더해 최종 모델을 만듭니다.
xgb_optuna_model = XGBClassifier(
    **study.best_params,
    random_state = 42,
    eval_metric = 'logloss'
)

# train 전체로 다시 학습
xgb_optuna_model.fit(X_train, y_train)

# test로 예측
xgb_optuna_pred = xgb_optuna_model.predict(X_test)
xgb_optuna_proba = xgb_optuna_model.predict_proba(X_test)[:, 1]

# 공통 평가 함수로 정리
xgb_optuna_result = evaluate_classification_model(
    model_name = 'XGBoost Optuna',
    y_true = y_test,
    y_pred = xgb_optuna_pred,
    y_proba = xgb_optuna_proba
)
xgb_optuna_result

{'model_name': 'XGBoost Optuna',
 'accuracy': 0.9736842105263158,
 'precision_malignant': 1.0,
 'recall_malignant': 0.9285714285714286,
 'f1_malignant': 0.9629629629629629,
 'roc_auc': 0.9923941798941799,
 'TN': np.int64(72),
 'FP': np.int64(0),
 'FN': np.int64(3),
 'TP': np.int64(39)}

In [70]:
# 세 결과를 하나의 표로 모읍니다.
xgb_tuning_results_df = pd.DataFrame([
    xgb_base_result,
    xgb_random_search_result,
    xgb_optuna_result
])

xgb_tuning_results_df

,model_name,accuracy,precision_malignant,recall_malignant,f1_malignant,roc_auc,TN,FP,FN,TP
0,XGBoost Base,0.964912,1.0,0.904762,0.950000,0.996693,72,0,4,38
1,XGBoost RandomizedSearchCV,0.982456,1.0,0.952381,0.975610,0.993056,72,0,2,40
2,XGBoost Optuna,0.973684,1.0,0.928571,0.962963,0.992394,72,0,3,39


In [74]:
# 다른모델: 로지스틱 회귀분석
logistic_model = Pipeline(
    steps = [
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(max_iter = 1000, random_state = 42))
    ]
)

logistic_model.fit(X_train, y_train)

logistic_pred = logistic_model.predict(X_test)
logistic_proba = logistic_model.predict_proba(X_test)[:, 1]

logistic_result = evaluate_classification_model(
    model_name = 'Logistic Regression',
    y_true = y_test,
    y_pred = logistic_pred,
    y_proba = logistic_proba
)
logistic_result

{'model_name': 'Logistic Regression',
 'accuracy': 0.9649122807017544,
 'precision_malignant': 0.975,
 'recall_malignant': 0.9285714285714286,
 'f1_malignant': 0.9512195121951219,
 'roc_auc': 0.996031746031746,
 'TN': np.int64(71),
 'FP': np.int64(1),
 'FN': np.int64(3),
 'TP': np.int64(39)}

In [75]:
# 다른모델: 랜덤포레스트
# 트리 200개로 구성
rf_model = RandomForestClassifier(
    n_estimators = 200,
    random_state = 42
)

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)
rf_proba = rf_model.predict_proba(X_test)[:,1]

rf_result = evaluate_classification_model(
    model_name = 'RandomForest',
    y_true = y_test,
    y_pred = rf_pred,
    y_proba = rf_proba
)

rf_result

{'model_name': 'RandomForest',
 'accuracy': 0.9649122807017544,
 'precision_malignant': 1.0,
 'recall_malignant': 0.9047619047619048,
 'f1_malignant': 0.95,
 'roc_auc': 0.994212962962963,
 'TN': np.int64(72),
 'FP': np.int64(0),
 'FN': np.int64(4),
 'TP': np.int64(38)}

In [76]:
# LightGBM 모델 (verbose = -1로 학습 로그 끔)
lgbm_model = LGBMClassifier(
    n_estimators = 200,
    learning_rate = 0.05,
    random_state = 42,
    verbose = -1
)

lgbm_model.fit(X_train, y_train)

lgbm_pred = lgbm_model.predict(X_test)
lgbm_proba = lgbm_model.predict_proba(X_test)[:, 1]

lgbm_result = evaluate_classification_model(
    model_name = 'LightGBM',
    y_true = y_test,
    y_pred = lgbm_pred,
    y_proba = lgbm_proba
)

lgbm_result